##### Importing required libraries

In [1]:
import openai             # For LLM interaction
import json               # For parsing LLM responses
import networkx as nx     # For creating and managing the graph data structure
import ipycytoscape       # For interactive in-notebook graph visualization
import ipywidgets         # For interactive elements
import pandas as pd       # For displaying data in tables
import os                 # For accessing environment variables (safer for API keys)
import math               # For basic math operations
import re                 # For basic text cleaning (regular expressions)
import warnings           # To suppress potential deprecation warnings

# Configure settings for better display and fewer warnings
warnings.filterwarnings('ignore', category=DeprecationWarning)
pd.set_option('display.max_rows', 100) # Show more rows in pandas tables
pd.set_option('display.max_colwidth', 150) # Show more text width in pandas tables

print("Libraries imported successfully.")

Libraries imported successfully.


##### Configuring the LLM Connection

In [2]:
'''
# Run one of these code in your terminal to set up the environment variables for the OpenAI API key and base URL.
# This is a safer way to handle sensitive information like API keys.
# Make sure to replace 'your_provider_api_key_here' with your actual API key.
# You can also set these variables in your environment directly or use a .env file with a library like python-dotenv.
 
# If using standard OpenAI
export OPENAI_API_KEY='your_openai_api_key_here'

# If using a local model like Ollama
export OPENAI_API_KEY='ollama' # Can be any non-empty string for Ollama
export OPENAI_API_BASE='http://localhost:11434/v1'

# If using another provider like Nebius AI
export OPENAI_API_KEY='your_provider_api_key_here'
export OPENAI_API_BASE='https://api.studio.nebius.com/v1/' # Example URL
'''

"\n# Run one of these code in your terminal to set up the environment variables for the OpenAI API key and base URL.\n# This is a safer way to handle sensitive information like API keys.\n# Make sure to replace 'your_provider_api_key_here' with your actual API key.\n# You can also set these variables in your environment directly or use a .env file with a library like python-dotenv.\n\n# If using standard OpenAI\nexport OPENAI_API_KEY='your_openai_api_key_here'\n\n# If using a local model like Ollama\nexport OPENAI_API_KEY='ollama' # Can be any non-empty string for Ollama\nexport OPENAI_API_BASE='http://localhost:11434/v1'\n\n# If using another provider like Nebius AI\nexport OPENAI_API_KEY='your_provider_api_key_here'\nexport OPENAI_API_BASE='https://api.studio.nebius.com/v1/' # Example URL\n"

In [9]:
# --- Define LLM Model ---
# Choose the model available at your configured endpoint.
# Examples: 'gpt-4o', 'gpt-3.5-turbo', 'llama3', 'mistral', 'deepseek-ai/DeepSeek-Coder-V2-Lite-Instruct', 'gemma'
llm_model_name = "deepseek-ai/DeepSeek-V3-0324" # <-- *** CHANGE THIS TO YOUR MODEL ***

print(f"Intended LLM model: {llm_model_name}")

Intended LLM model: deepseek-ai/DeepSeek-V3-0324


In [10]:
# --- Retrieve Credentials ---
api_key = os.getenv("OPENAI_API_KEY")
base_url = os.getenv("OPENAI_API_BASE") # Will be None if not set (e.g., for standard OpenAI)

print(f"Retrieved API Key: {'Set' if api_key else 'Not Set'}")
print(f"Retrieved Base URL: {base_url if base_url else 'Not Set (will use default OpenAI)'}")

Retrieved API Key: Set
Retrieved Base URL: https://api.studio.nebius.com/v1/


In [11]:
# --- Validate Key and Initialize Client --- 
if not api_key:
    print("Error: OPENAI_API_KEY environment variable not set or key not provided directly.")
    print("Please set the environment variable (or uncomment/edit the test lines) and restart the kernel.")
    raise SystemExit("API Key configuration failed.")
else:
    try:
        client = openai.OpenAI(
            base_url=base_url, # Pass None if not set, client handles default
            api_key=api_key
        )
        print("OpenAI client initialized successfully.")
    except Exception as e:
        print(f"Error initializing OpenAI client: {e}")
        print("Check your API key, base URL (if used), and network connection.")
        raise SystemExit("LLM client initialization failed.")

OpenAI client initialized successfully.


In [12]:
# --- Define LLM Call Parameters ---
llm_temperature = 0.0 # Lower temperature for more deterministic, factual output. 0.0 is best for extraction.
llm_max_tokens = 4096 # Max tokens for the LLM response (adjust based on model limits)

print(f"LLM Temperature set to: {llm_temperature}")
print(f"LLM Max Tokens set to: {llm_max_tokens}")

LLM Temperature set to: 0.0
LLM Max Tokens set to: 4096


##### Defining Input Text